# Notebook 17 — Precompute app data from the authoritative fold-1 model

This notebook loads the trained fold-1 MT-TrajNet model and precomputes everything
the Streamlit demo needs (predictions, uncertainty, calibrated intervals) for the
fold-1 test batches plus the two out-of-distribution codes. It does not retrain and
does not modify any result files; the reported numbers stay unchanged. The model is
verified against the authoritative fold-1 RMSE before anything is saved.

In [1]:
import numpy as np, pickle, torch, torch.nn as nn, random, hashlib, json, time
from sklearn.metrics import mean_squared_error

SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False
device="cuda" if torch.cuda.is_available() else "cpu"

DATA="/kaggle/input/datasets/arpitjoshua/mt-trajnet-thesis-data/kaggle_upload"
with open(DATA+"/assembled_trajectories.pkl","rb") as fh:
    data=pickle.load(fh)
X_traj=data["X_traj"]; y_arr=data["y_arr"]; groups=data["groups"]

h=hashlib.md5()
with open(DATA+"/assembled_trajectories.pkl","rb") as fh:
    for c in iter(lambda:fh.read(1<<20),b""): h.update(c)
assert h.hexdigest()=="7d7fc76be1e4940198b76d9d0797a3a9","stop: data is not the authoritative version"
print("data verified:",len(X_traj),"batches,",len(np.unique(groups)),"product codes")

data verified: 1005 batches, 25 product codes


## Model definition, folds, and verification

This rebuilds the exact MT-TrajNet architecture (a dilated causal TCN encoder with
four evidential regression heads) and the stratified fold assignment used in the
main experiments. It then loads the saved fold-1 weights and confirms they
reproduce the authoritative fold-1 RMSE before any app data is computed. If the
numbers do not match, the notebook stops here.

In [2]:
import torch.nn.functional as F

STRIDE=2; MAXLEN=6000
targets=["dissolution_av","tbl_av_hardness","tbl_rsd_weight","fct_tensile"]

def prep_batch(traj_list,mean,std,maxlen=MAXLEN):
    out=[]
    for a in traj_list:
        a=a[::STRIDE]; a=(a-mean)/std
        if len(a)<maxlen:
            a=np.vstack([a,np.zeros((maxlen-len(a),a.shape[1]),dtype="float32")])
        else:
            a=a[:maxlen]
        out.append(a)
    return torch.tensor(np.array(out,dtype="float32")).transpose(1,2)

fold_codes=[[1,2,4,6,7,9,13,18,25],[10,11,17,19,22,24],[3,5,8,12,14,16,20,21,23]]
fold_test_idx=[np.where(np.isin(groups,fc))[0] for fc in fold_codes]

class TCNBlock(nn.Module):
    def __init__(self,in_ch,out_ch,dilation,kernel=3,dropout=0.1):
        super().__init__()
        pad=(kernel-1)*dilation
        self.conv1=nn.Conv1d(in_ch,out_ch,kernel,padding=pad,dilation=dilation)
        self.conv2=nn.Conv1d(out_ch,out_ch,kernel,padding=pad,dilation=dilation)
        self.relu=nn.ReLU(); self.drop=nn.Dropout(dropout); self.pad=pad
        self.down=nn.Conv1d(in_ch,out_ch,1) if in_ch!=out_ch else None
    def forward(self,x):
        res=x if self.down is None else self.down(x)
        out=self.conv1(x)[:,:,:-self.pad]; out=self.drop(self.relu(out))
        out=self.conv2(out)[:,:,:-self.pad]; out=self.drop(self.relu(out))
        return self.relu(out+res)

class TCNEncoder(nn.Module):
    def __init__(self,in_ch=10,hidden=128,dropout=0.1):
        super().__init__()
        self.b1=TCNBlock(in_ch,hidden,1,dropout=dropout); self.b2=TCNBlock(hidden,hidden,2,dropout=dropout)
        self.b3=TCNBlock(hidden,hidden,4,dropout=dropout); self.b4=TCNBlock(hidden,hidden,8,dropout=dropout)
    def forward(self,x):
        x=self.b1(x);x=self.b2(x);x=self.b3(x);x=self.b4(x); return x.mean(dim=2)

class EvidentialHead(nn.Module):
    def __init__(self,in_dim,hidden=64,dropout=0.1):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(in_dim,hidden),nn.ReLU(),nn.Dropout(dropout),nn.Linear(hidden,4))
    def forward(self,x):
        out=self.net(x)
        gamma=out[:,0]
        nu=F.softplus(out[:,1]).clamp(min=1e-2,max=1e3)
        alpha=F.softplus(out[:,2]).clamp(min=1e-2,max=1e3)+1.0
        beta=F.softplus(out[:,3]).clamp(min=1e-2,max=1e3)
        return gamma,nu,alpha,beta

class MTTrajNet(nn.Module):
    def __init__(self,in_ch=10,hidden=128,n_targets=4,dropout=0.1):
        super().__init__()
        self.encoder=TCNEncoder(in_ch,hidden,dropout)
        self.heads=nn.ModuleList([EvidentialHead(hidden,64,dropout) for _ in range(n_targets)])
    def forward(self,x):
        z=self.encoder(x)
        gs=[h(z) for h in self.heads]
        gamma=torch.stack([g[0] for g in gs],dim=1)
        nu=torch.stack([g[1] for g in gs],dim=1)
        alpha=torch.stack([g[2] for g in gs],dim=1)
        beta=torch.stack([g[3] for g in gs],dim=1)
        return gamma,nu,alpha,beta

BUNDLE="/kaggle/input/datasets/arpitjoshua/mt-trajnet-thesis-data/mttrajnet_fold1.pt"
b=torch.load(BUNDLE,map_location=device,weights_only=False)
model=MTTrajNet().to(device); model.load_state_dict(b["state"]); model.eval()
xmean,xstd,ymean,ystd=b["xmean"],b["xstd"],b["ymean"],b["ystd"]

def pred_std(idx):
    Xb=prep_batch([X_traj[i] for i in idx],xmean,xstd)
    gg=[];ss=[]
    with torch.no_grad():
        for i in range(0,len(Xb),16):
            g,nu,al,be=model(Xb[i:i+16].to(device))
            v=be/(nu*(al-1.0))
            gg.append(g.cpu().numpy()); ss.append(torch.sqrt(v).cpu().numpy())
    return np.vstack(gg)*ystd+ymean, np.vstack(ss)*ystd

te=fold_test_idx[0]
p_te,s_te=pred_std(te)
rmse={targets[k]:round(float(np.sqrt(mean_squared_error(y_arr[te,k],p_te[:,k]))),3) for k in range(4)}
p4,s4=pred_std(np.where(groups==4)[0])
print("fold-1 RMSE:",rmse,"   (must be 3.344/7.9/0.77/0.404)")
print("code 4 hardness std median:",round(float(np.median(s4[:,1])),3),"   (must be ~69.9)")

fold-1 RMSE: {'dissolution_av': 3.344, 'tbl_av_hardness': 7.9, 'tbl_rsd_weight': 0.77, 'fct_tensile': 0.404}    (must be 3.344/7.9/0.77/0.404)
code 4 hardness std median: 69.947    (must be ~69.9)


## Precompute the demo data

For the fold-1 test batches and the two unseen product codes (15 and 25), this
computes the model's predictions, its epistemic and aleatoric uncertainty, and
the calibrated 90% prediction intervals using the fold-1 calibration scales. The
results are written to a single file that the demo application reads directly, so
the app itself runs no model and needs no GPU.

In [3]:
scale_fold1={"dissolution_av":0.228,"tbl_av_hardness":0.327,"tbl_rsd_weight":0.572,"fct_tensile":0.214}
scale=np.array([scale_fold1[t] for t in targets])
z90=1.6448536269514722

def build_rows(idx):
    pred,std=pred_std(idx)
    std_cal=std*scale
    rows=[]
    for j,i in enumerate(idx):
        rows.append({
            "batch_id":int(i),"code":int(groups[i]),
            "pred":{targets[k]:round(float(pred[j,k]),3) for k in range(4)},
            "true":{targets[k]:round(float(y_arr[i,k]),3) for k in range(4)},
            "std":{targets[k]:round(float(std_cal[j,k]),3) for k in range(4)},
            "pi90_low":{targets[k]:round(float(pred[j,k]-z90*std_cal[j,k]),3) for k in range(4)},
            "pi90_high":{targets[k]:round(float(pred[j,k]+z90*std_cal[j,k]),3) for k in range(4)},
        })
    return rows

idx_test=fold_test_idx[0]; idx_c15=np.where(groups==15)[0]; idx_c25=np.where(groups==25)[0]
app={"targets":targets,
     "channels":["fom","cyl_pre","tbl_speed","pre_comp","SREL","tbl_fill","cyl_main","ejection","main_comp","stiffness"],
     "fold1_scale":scale_fold1,
     "batches":build_rows([i for i in idx_test if groups[i]!=25])+build_rows(idx_c15)+build_rows(idx_c25)}
with open("/kaggle/working/app_data.json","w") as fh: json.dump(app,fh)

def med(code,t):
    v=[r["std"][t] for r in app["batches"] if r["code"]==code]; return round(float(np.median(v)),2)
print("total batches:",len(app["batches"]),"| codes:",sorted(set(r["code"] for r in app["batches"])))
print("median calibrated hardness std -> code 1:",med(1,"tbl_av_hardness"),
      "| code 25:",med(25,"tbl_av_hardness"),"| code 15:",med(15,"tbl_av_hardness"))

total batches: 378 | codes: [1, 2, 4, 6, 7, 9, 13, 15, 18, 25]
median calibrated hardness std -> code 1: 7.23 | code 25: 6.85 | code 15: 25.0
